# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Type: Ranking / Scoring**, built on top of a binary classifier.

My Lane 2 output is a ranked review queue -- "which pages should an editor look at first" -- which is a
ranking/scoring problem, not a plain classification problem on its own (per the `framing-ml-problems`
mapping: "which ones first?" -> ranking/scoring, target = a priority score, metric = precision@K).
Underneath the ranking, a binary classifier supplies the probability that feeds the score: the starter
pipeline trains a classifier on `is_declining_label` and uses `predict_proba` as the ranking signal, then
blends it with a transparent baseline score into one `final_refresh_score`. So the end-to-end task is
scoring/ranking; classification is the engine inside it, not the deliverable itself.

In [1]:
import pandas as pd, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df["trend_direction"].value_counts())
print(f"\nis_declining_label rate (trend_direction == 'down'): {(df['trend_direction'].str.lower()=='down').mean():.3f}")
print("This confirms the label is a CURRENT-window bucket, not a future-measured event -- a proxy, as stated above.")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label rate (trend_direction == 'down'): 0.542
This confirms the label is a CURRENT-window bucket, not a future-measured event -- a proxy, as stated above.


## 2. Target or proxy

**Proxy label, not yet a true observed future outcome:** `is_declining_label = (trend_direction == "down")`.
`trend_direction` itself is a bucket computed from `trend_pct`, i.e. it describes the CURRENT window's
trend, not a confirmed future decline. That makes it a defined-rule proxy -- useful for wiring up the
pipeline end to end, but the lane guide is explicit that a stronger capstone should move to a genuinely
**observed future outcome**, e.g. "features from the prior 90 days -> decline over the next 30 days," once
I bring in the warehouse's daily table in later weeks. I'm naming this limitation now rather than
discovering it in Week 5.

In [2]:
res = json.load(open("outputs/model_results.json"))
base_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50 = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule   Precision@50: {base_p50:.3f}  (~{round(base_p50*50)} of the top 50 right)")
print(f"Random forest   Precision@50: {rf_p50:.3f}  (~{round(rf_p50*50)} of the top 50 right)")
print("Both scored the same way (client-holdout) -- this is the number that decides whether my lane is worth shipping.")


Baseline rule   Precision@50: 0.240  (~12 of the top 50 right)
Random forest   Precision@50: 0.740  (~37 of the top 50 right)
Both scored the same way (client-holdout) -- this is the number that decides whether my lane is worth shipping.


## 3. Success metric

**Precision@K** (K=20 and K=50), not accuracy or ROC-AUC alone. An editor only reviews a fixed number of
pages per sprint, so what matters is: of the top K pages the ranking puts first, how many are actually
worth reviewing? That is exactly what Precision@K measures, and it is the metric the starter pipeline
itself reports (see the code cell below: baseline rule vs random forest, both measured the same way with
client-holdout validation). "Good" for my lane means beating the transparent baseline rule's Precision@50
by a real, validated margin -- not just having a high-sounding overall accuracy number.

In [3]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"rows: {len(df):,}   unique content_id: {df['content_id'].nunique():,}   unique client_id: {df['client_id'].nunique()}")
print("content_id is unique per row -> one row = one page's 90-day snapshot for one client.\n")

unit_of_analysis = df[["content_id", "client_id", "content_type", "impressions_90d",
                        "days_since_last_update", "avg_position", "ctr", "trend_direction"]].head(5)
unit_of_analysis


rows: 30,000   unique content_id: 30,000   unique client_id: 32
content_id is unique per row -> one row = one page's 90-day snapshot for one client.



,content_id,client_id,content_type,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,20,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,25,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,20,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,22,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,14,44.0,0.13,down


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page) for one client, summarized over a trailing 90-day window.**
`content_id` is the row-level key; `client_id` groups rows for holdout splits. The code cell below loads
the starter slice and confirms `content_id` is unique per row (one page, one snapshot row) and shows what
that row actually looks like.

In [4]:
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"Single hand rule (stale AND visible) fires on {len(stale_visible)} of {len(df):,} pages "
      f"({100*len(stale_visible)/len(df):.1f}%) -- far too narrow to rank a whole inventory on its own.")
print(f"\nOn the SAME ranking task, the learned model reaches Precision@50 = {rf_p50:.3f} vs the "
      f"rule's {base_p50:.3f} ({rf_p50/base_p50:.1f}x) -- the combination of signals the model finds is doing"
      " real work the single rule cannot.")


Single hand rule (stale AND visible) fires on 17 of 30,000 pages (0.1%) -- far too narrow to rank a whole inventory on its own.

On the SAME ranking task, the learned model reaches Precision@50 = 0.740 vs the rule's 0.240 (3.1x) -- the combination of signals the model finds is doing real work the single rule cannot.


## 5. Why ML beats a fixed rule here

A single hand-written rule like "stale AND visible" only fires on a tiny sliver of the inventory (see
below) -- it can't rank the other 99%+ of pages at all, let alone in the right order. The real signal is
spread across several correlated-but-not-identical measurements (freshness, visibility, position, CTR,
content depth) whose *combination and relative weighting* is what actually separates a page worth
reviewing from one that isn't -- and that weighting differs by client and content type. That is precisely
the kind of "real but too messy to hand-write" pattern the `framing-ml-problems` skill says ML earns its
place on. The evidence isn't hypothetical: on this same starter slice, the learned model already
outperforms the rule by a validated margin (client-holdout, not in-sample) -- see the numbers below.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.